# 1.2.1 NumPy

Vectorized operations, broadcasting, linear algebra, indexing — applied to California Housing data.

In [ ]:
import numpy as np, csv, urllib.request
from pathlib import Path

DATA = Path('data'); DATA.mkdir(exist_ok=True)
dest = DATA/'cal_housing.csv'
if not dest.exists():
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/scikit-learn/california-housing/resolve/main/cal_housing.csv', dest)

with open(dest, newline='') as f: rows = list(csv.DictReader(f))
cols = list(rows[0].keys())
X = np.array([[float(r[c]) for c in cols[:-1]] for r in rows])
y = np.array([float(r[cols[-1]]) for r in rows])
print(f'X: {X.shape}, y: {y.shape}, dtype: {X.dtype}')

In [ ]:
# Vectorized stats
print('Feature stats (mean, std, min, max):')
for i, col in enumerate(cols[:-1]):
    print(f'  {col:<15} mean={X[:,i].mean():>8.3f}  std={X[:,i].std():>7.3f}  min={X[:,i].min():>7.3f}  max={X[:,i].max():>8.3f}')

In [ ]:
# Broadcasting: z-score standardize
mean = X.mean(axis=0)
std  = X.std(axis=0) + 1e-8
X_std = (X - mean) / std
print('Standardized X mean:', np.round(X_std.mean(axis=0), 3))
print('Standardized X std :', np.round(X_std.std(axis=0), 3))

In [ ]:
# Linear algebra: OLS via normal equations
X_bias = np.column_stack([np.ones(len(X_std)), X_std])
w = np.linalg.solve(X_bias.T @ X_bias, X_bias.T @ y)
y_pred = X_bias @ w
r2 = 1 - np.sum((y-y_pred)**2)/np.sum((y-y.mean())**2)
print(f'OLS R² = {r2:.4f}')

In [ ]:
# Visualization
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,2, figsize=(12,4))
axes[0].scatter(X[:,0], y, alpha=0.2, s=3)
axes[0].set_xlabel('MedInc'); axes[0].set_ylabel('MedHouseVal')
axes[0].set_title('Income vs House Value')
axes[1].hist(y, bins=50, edgecolor='white')
axes[1].set_xlabel('MedHouseVal'); axes[1].set_title('House Value Distribution')
plt.tight_layout(); plt.show()